In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from scipy.stats import uniform, loguniform, norm
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from scipy.optimize import minimize
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

## **Function 2** - Mystery ML model

- Model takes two numbers as input and returns a log-likelihood score.
  - **Goal** - maximise that score, but each output is noisy, and depending on where you start, you might get stuck in a local optimum.

- **Method of tackling this problem** - Bayesian optimisation, which selects the next inputs based on what it has learned so far.
  - I will aim to balance exploration with exploitation, making it well suited to noisy outputs and complex functions with many local peaks.

In [2]:
# Load current cumulative dataset (original + prior weekly updates) for Function 2
X = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_2/initial_inputs.npy')
Y = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_2/initial_outputs.npy')

# New data from Week 4 (Function 2)
X_w4_new_point = np.array([0.688961, 0.010598], dtype=np.float64)
Y_w4_new_point = np.array([0.5596611899528867], dtype=np.float64)

# Append the new data point
X_updated = np.vstack((X, X_w4_new_point.reshape(1, -1)))

# Remove duplicate X rows (if point already exists)
X_unique, unique_indices = np.unique(X_updated, axis=0, return_index=True)

# Build Y and keep matching rows only
Y_all = np.append(Y, Y_w4_new_point)
Y_updated = Y_all[unique_indices]

# Save updated arrays
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_2/initial_inputs.npy', X_unique)
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_2/initial_outputs.npy', Y_updated)

# Show updated arrays
print("Updated Inputs (X) - Function 2: ", X_unique)
print("Updated Outputs (Y) - Function 2: ", Y_updated)

Updated Inputs (X) - Function 2:  [[0.14269907 0.34900513]
 [0.33864816 0.21386725]
 [0.34174959 0.02869772]
 [0.43816606 0.68501826]
 [0.45464714 0.29045518]
 [0.57771284 0.77197318]
 [0.638512   0.01429   ]
 [0.66579958 0.12396913]
 [0.688961   0.010598  ]
 [0.70263656 0.9265642 ]
 [0.84527543 0.71112027]
 [0.871256   0.999348  ]
 [0.87779099 0.7786275 ]
 [0.998312   0.002851  ]]
Updated Outputs (Y) - Function 2:  [-0.06562362 -0.01385762  0.03874902  0.24461934  0.21496451  0.02310555
  0.29668333  0.53899612  0.55966119  0.61120522  0.29399291  0.05410069
  0.42058624 -0.08088786]


### **Output Analysis**

- In Week 4, I queried [0.688961, 0.010598] and received an output of 0.5597. This is a massive leap forward from our Week 3 output of -0.0541 at [0.871256, 0.999348].

- **Real-World Implication:** I have successfully crossed the threshold from a negative log-likelihood (where the model parameters were a poor fit) to a positive one. 
    - This indicates I am finally in a region where the parameters are starting to align well with the mystery model's behavior.

- **Model Clues:** The jump of over 0.6 units suggests the objective function has a relatively steep gradient in this part of the search space. 
    - However, since the output is known to be noisy, we need to determine if this 0.5597 is a stable peak or if we've just caught a "lucky" noise spike near a local optimum.


### **Baysian Optimisation**: Gaussian Process Regressor

I am sticking with the Matern kernel ($nu=1.5$) because its "rougher" nature is better suited for a noisy log-likelihood surface than a perfectly smooth RBF kernel. 

I am also ensuring ARD (Automatic Relevance Determination) is active by allowing independent length scales for each input dimension. 
- This is critical because the "Mystery ML" parameters might have very different sensitivities—one input might need tiny adjustments while the other requires large swings to change the score.

In [3]:
# 1. Update the Kernel for Function 2 (Mystery ML Model)
# We use nu=1.5 for a rugged surface and WhiteKernel to estimate the noise level
kernel = Matern(length_scale=[0.1, 0.1], nu=1.5) + WhiteKernel(noise_level=0.1)

# 2. Define the Gaussian Process Regressor
# normalize_y=True is vital as log-likelihoods can span different orders of magnitude
model = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=25,
    normalize_y=True)

# 3. Fit the model to our specific Week 3 and Week 4 data
model.fit(X_unique, Y_updated)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",Matern(length...ise_level=0.1)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",25
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",True
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`G

### **Acquisition Function**

- Since I've found a promising "warm" zone with our 0.5597 result, the goal is now to exploit this success while still allowing the model to scout for a better global maximum. I am using the Upper Confidence Bound (UCB) with a $\beta$ of 2.0.

- A $\beta$ of 2.0 is a balanced choice: it gives enough weight to the predicted mean ($\mu$) to keep us near our current best area, but provides a 2-standard-deviation "bonus" for uncertainty ($\sigma$). 
    - This prevents us from getting stuck in a local peak if there is a much higher "mountain" of log-likelihood just a short distance away in the unexplored space.

In [7]:
def upper_confidence_bound(X, model, beta=2.0):
    mu, sigma = model.predict(X, return_std=True)
    return mu + beta * sigma

beta = 2.0
eps = 1e-3  # trying this to avoid exact 0 and 1
n = 100

axis = np.linspace(eps, 1 - eps, n)
x1, x2 = np.meshgrid(axis, axis, indexing="xy")
x_grid = np.column_stack([x1.ravel(), x2.ravel()])

ucb_values = upper_confidence_bound(x_grid, model, beta=beta)
next_query = x_grid[np.argmax(ucb_values)]

print(f"Strategic Week 5 Query (Function 2): {next_query[0]:.6f}-{next_query[1]:.6f}")

Strategic Week 5 Query (Function 2): 0.736899-0.001000
